Bing Chen 

ST 554 Project Part 2 

3/19/2026

# Introduction 

In this project, we developed a PySpark-based data validation class named `SparkDataCheck`. Our class is designed to facilitate data validation and exploration. After defining the class and its associated methods, we will apply it to some real-world dataset here to perform various data quality checks, such as missing value detection, numerical range validation, and etc. 


Now, let's import the class and start a spark session. 

In [10]:
from SparkDataCheck import SparkDataCheck
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd


spark = SparkSession.builder \
    .appName("SparkDataCheck") \
    .getOrCreate()



# From csv 

Next, let's create an instance of the class from a `.csv` file. 

In [11]:
air = SparkDataCheck.from_csv(
    spark,
    "air.csv"
)

air.df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-20 21:00:00|   2.2|       137

26/03/20 16:37:49 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


Using this air quality data, let's see some examples of how to use methods in this class. Recall that in this dataset, missing values are tagged with -200 value, so let's replace them with `None` first. 

In [12]:
air.df = air.df.na.replace(-200, None)

## Check if a numeric variable is within a specified range. 

Next, we might want to check if `PT08.S1(CO)` values specifically are all above 0. 


In [13]:
air.within_range("PT08.S1(CO)", lower=0, upper = None).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|                true|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|                true|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

26/03/20 16:37:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


Let's see if the values are all below 2000. 

In [14]:
air.within_range("PT08.S1(CO)", lower=None, upper = 1300).df.show()

26/03/20 16:37:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|                true|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

How about between 0 to 1500?

In [15]:
air.within_range("PT08.S1(CO)", lower=0, upper = 10).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               false|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        114

26/03/20 16:37:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


What if we accidentally passed through a string variable? 

In [16]:
air.within_range("Date", lower=0, upper = None).df.show()

Column 'Date' is not numeric. 
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               false|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       1402|      88|     9.0|  

26/03/20 16:37:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


Nice! It prints out message noting that `Date` is not a numeric column and returns the original data frame. 

What if we forgot to pass on any bounds?

In [17]:
air.within_range("PT08.S1(CO)").df.show()

At least one of 'lower' or 'upper' must be provided. 
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               false|
|  2|3/10/2004|2026-03-20 20:00:00|   2.2|       14

26/03/20 16:37:51 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


So, it prints out that at least one bound need to be supplied. Great! 

## Check if a categorical variable is within specified levels. 

Say we want to classify `PT08.S1(CO)` into 3 levels to allow users to quickly understand if air quality is "Safe" or "Hazardous" without knowing technical sensor specs. Consider the 3 ranks below: 

 - Low: Readings below 970
 - Mid: Readings between 970 and 1,180
 - High: Readings above 1,180
 
 So, let's create a new variable for the transformed `PT08.S1(CO)`. 


In [18]:
air.df = air.df.withColumn(
    "PT08_S1(CO)_cat",
    F.when(F.col("`PT08.S1(CO)`") < 970, "Low")
     .when((F.col("`PT08.S1(CO)`") >= 970) & (F.col("`PT08.S1(CO)`") <= 1180), "Mid")
     .when(F.col("`PT08.S1(CO)`") > 1180, "High")
     .otherwise(None)
)

Let's quickly check the results. 

In [19]:
air.df.select("`PT08.S1(CO)`","PT08_S1(CO)_cat").show(10)

+-----------+---------------+
|PT08.S1(CO)|PT08_S1(CO)_cat|
+-----------+---------------+
|       1360|           High|
|       1292|           High|
|       1402|           High|
|       1376|           High|
|       1272|           High|
|       1197|           High|
|       1185|           High|
|       1136|            Mid|
|       1094|            Mid|
|       1010|            Mid|
+-----------+---------------+
only showing top 10 rows


Looks like it's working correctly. Let's try out our method to see if all values are within the levels. 

In [20]:
air.within_levels("PT08.S1(CO)", levels = ["Low", "Mid", "High"]).df.show()

Column 'PT08.S1(CO)' is not a string column.
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               fal

26/03/20 16:37:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


Oops! Accidentally passed through the numeric column of `PT08.S1(CO)` instead of the categorical column we just created! So, let's do that again. 

In [21]:
air.within_levels("PT08_S1(CO)_cat", levels = ["Low", "Mid", "High"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|                     true|
|  1|3/10/2004|2026-03-20 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92| 

26/03/20 16:37:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


Looks good! Let's try another variable.  We can `Absolute Humidity` into 

- Dry: AH < 0.8  
- Optimal: 0.8 ≤ AH ≤ 1.5  
- Humid: AH > 1.5

In [22]:
air.df = air.df.withColumn(
    "abs_humidity_cat",
    F.when(F.col("AH") < 0.8, "Dry")
     .when((F.col("AH") >= 0.8) & (F.col("AH") <= 1.5), "Optimal")
     .when(F.col("AH") > 1.5, "Humid")
     .otherwise(None)
)

In [23]:
air.within_levels("abs_humidity_cat", levels = ["Optimal", "Humid"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|           

26/03/20 16:37:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


Why is there so many `false`? Oops! Accidentally missed out a level! 

In [24]:
air.within_levels("abs_humidity_cat", levels = ["Dry", "Optimal", "Humid"]).df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|               false|           High|           

26/03/20 16:37:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


Now it's looking good. 

## Check Missingness 

When we first look at a dataset, we always want to know how complete is the data, are there any missing values, how are they coded? From the data description, we know that missing values are coded -200 and we already replaced that with `None`. So now, let's check where a value is missing for some variables. 

In [25]:
air.is_missing("`PT08.S1(CO)`").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|  

26/03/20 16:37:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


In [26]:
air.is_missing("AH").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+
|  0|3/10/2004|2026-03-20 18:00:00|   2.6|       1360|     150|    11.9|         1046|  

26/03/20 16:37:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


In [27]:
air.is_missing("NMHC(GT)").df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|NMHC(GT)_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+
|  0|3/10/2004|2026-03-20 18

26/03/20 16:37:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


In [28]:
air.is_missing("Date").df.show()

26/03/20 16:37:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-------------------+---------------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|PT08.S1(CO)_in_range|PT08_S1(CO)_cat|PT08_S1(CO)_cat_in_levels|abs_humidity_cat|abs_humidity_cat_in_levels|`PT08.S1(CO)`_is_missing|AH_is_missing|NMHC(GT)_is_missing|Date_is_missing|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------------+---------------+-------------------------+----------------+--------------------------+------------------------+-------------+-----------------

Looks good!

## Report the min and max of a given numeric column 

We previously checked if `PT08.S1(CO)` is within a specific range of interest. But what if we have no idea where the values should fall? This method is helpful to tell us this information. 


In [29]:
air.min_max("PT08.S1(CO)").show()

+---------------+---------------+
|PT08.S1(CO)_min|PT08.S1(CO)_max|
+---------------+---------------+
|            647|           2040|
+---------------+---------------+



In [30]:
air.min_max("PT08_S1(CO)_cat")

Column 'PT08_S1(CO)_cat' is not numeric. Please give a numeric column.


We can't use this method on string variables! 

I want to see the min and max of `PT08.S1(CO)` for each levels `Absolute Humidity`. 

In [31]:
air.min_max("PT08.S1(CO)", "abs_humidity_cat").show()

+----------------+---------------+---------------+
|abs_humidity_cat|PT08.S1(CO)_min|PT08.S1(CO)_max|
+----------------+---------------+---------------+
|            NULL|           NULL|           NULL|
|           Humid|            760|           1915|
|             Dry|            647|           1973|
|         Optimal|            704|           2040|
+----------------+---------------+---------------+



We can simply see that min and max for all numeric variables! 

In [32]:
air.min_max().show()

26/03/20 16:37:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/20 16:37:56 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


+-------+-------+----------+----------+---------------+---------------+------------+------------+------------+------------+-----------------+-----------------+-----------+-----------+----------------+----------------+-----------+-----------+----------------+----------------+---------------+---------------+-----+-----+------+------+------+------+
|_c0_min|_c0_max|CO(GT)_min|CO(GT)_max|PT08.S1(CO)_min|PT08.S1(CO)_max|NMHC(GT)_min|NMHC(GT)_max|C6H6(GT)_min|C6H6(GT)_max|PT08.S2(NMHC)_min|PT08.S2(NMHC)_max|NOx(GT)_min|NOx(GT)_max|PT08.S3(NOx)_min|PT08.S3(NOx)_max|NO2(GT)_min|NO2(GT)_max|PT08.S4(NO2)_min|PT08.S4(NO2)_max|PT08.S5(O3)_min|PT08.S5(O3)_max|T_min|T_max|RH_min|RH_max|AH_min|AH_max|
+-------+-------+----------+----------+---------------+---------------+------------+------------+------------+------------+-----------------+-----------------+-----------+-----------+----------------+----------------+-----------+-----------+----------------+----------------+---------------+-------------

In [33]:
air.min_max(col_name = None, group_by = "abs_humidity_cat").show()

26/03/20 16:37:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , AH
 Schema: _c0, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-bchen29@ncsu.edu/ST-554-Project2/Part2/air.csv


+----------------+-------+-------+----------+----------+---------------+---------------+------------+------------+------------+------------+-----------------+-----------------+-----------+-----------+----------------+----------------+-----------+-----------+----------------+----------------+---------------+---------------+-----+-----+------+------+------+------+
|abs_humidity_cat|_c0_min|_c0_max|CO(GT)_min|CO(GT)_max|PT08.S1(CO)_min|PT08.S1(CO)_max|NMHC(GT)_min|NMHC(GT)_max|C6H6(GT)_min|C6H6(GT)_max|PT08.S2(NMHC)_min|PT08.S2(NMHC)_max|NOx(GT)_min|NOx(GT)_max|PT08.S3(NOx)_min|PT08.S3(NOx)_max|NO2(GT)_min|NO2(GT)_max|PT08.S4(NO2)_min|PT08.S4(NO2)_max|PT08.S5(O3)_min|PT08.S5(O3)_max|T_min|T_max|RH_min|RH_max|AH_min|AH_max|
+----------------+-------+-------+----------+----------+---------------+---------------+------------+------------+------------+------------+-----------------+-----------------+-----------+-----------+----------------+----------------+-----------+-----------+------------

## Count the number of each levels for a given string variable

We may be interested in how many observations fall under each `PT08.S1(CO)` category we created.

In [34]:
air.count_level("PT08_S1(CO)_cat").show()

+---------------+-----+
|PT08_S1(CO)_cat|count|
+---------------+-----+
|           NULL|  366|
|           High| 2789|
|            Low| 2819|
|            Mid| 3383|
+---------------+-----+



Most of our data falls in to the Mid category! 

Let's try passing 2 variables. 

In [35]:
air.count_level("PT08_S1(CO)_cat", "abs_humidity_cat").show()

+---------------+----------------+-----+
|PT08_S1(CO)_cat|abs_humidity_cat|count|
+---------------+----------------+-----+
|           NULL|            NULL|  366|
|           High|             Dry|  659|
|           High|           Humid|  470|
|           High|         Optimal| 1660|
|            Low|             Dry| 1060|
|            Low|           Humid|  312|
|            Low|         Optimal| 1447|
|            Mid|             Dry| 1019|
|            Mid|           Humid|  536|
|            Mid|         Optimal| 1828|
+---------------+----------------+-----+



The count for each combination of levels are outputted! 

We can also use this function to count how many missing values we have for a variable. Recall that we created `AH_is_missing` variable. We can convert this column to a string and use this method.  

In [36]:
air.df = air.df.withColumn(
    "AH_is_missing_str",
    F.col("AH_is_missing").cast("string")
)

air.count_level("AH_is_missing_str").show()

+-----------------+-----+
|AH_is_missing_str|count|
+-----------------+-----+
|            false| 8991|
|             true|  366|
+-----------------+-----+



We have 366 missing values! 

If we just pass `AH_is_missing`, our program won't accept it! Because we set up our code to only accept string variables, and this is a Boolean column. 

In [37]:
air.count_level("AH_is_missing")

Column 'AH_is_missing' is a numeric column. Please provide a string column!


# From Pandas

Now let's try reading the data in from `pandas`! 

In [38]:
air2 = SparkDataCheck.from_pd(
    spark,
    pd.read_csv("air.csv")
)

air2.df.show()

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

Let's see what the min and max is if we don't recode the missing values for `CO(GT)`. 

In [39]:
air2.min_max("CO(GT)").show()

+----------+----------+
|CO(GT)_min|CO(GT)_max|
+----------+----------+
|    -200.0|      11.9|
+----------+----------+



# Part 2 

Let's read in the NFL data to do some basic analysis using spark. We'll need to import `Pandas on Spark`.

In [41]:
import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False)


nfl = pd.read_csv("weekly_nfl_data.csv")
nfl_df = ps.from_pandas(nfl)


/home/jupyter-bchen29@ncsu.edu/.local/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


See the first 5 rows of this data set. 


In [42]:
nfl_df.head()


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Check the column names 

In [43]:
nfl_df.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

We are only interested in looking at QB stats for the seasons 2005 to 2023. So, let's subset the data accordingly and only include columns `player_display_name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`. 

In [46]:
qb = nfl_df[(nfl_df["position"] == "QB") &
              (nfl_df["season_type"] == "REG") &
              (nfl_df["season"] >= 2005) &
              (nfl_df["season"] <= 2023)
             ][["player_display_name", "season", "week",
                "completions", "attempts", "passing_yards",
                "passing_tds", "interceptions"
               ]]


/home/jupyter-bchen29@ncsu.edu/.local/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


We want to know for each player and season combination, the sum and mean of each of the statistical quantities. 

In [50]:
qb_summary = (
    qb.groupby(["player_display_name", "season"])[
        ["completions", "attempts", "passing_yards", "passing_tds", "interceptions"]
    ]
    .agg(["sum", "mean"])
)
qb_summary.head()

/home/jupyter-bchen29@ncsu.edu/.local/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


completions            attempts            passing_yards             passing_tds           interceptions          
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                   
Jeff Blake          2005             8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.000000
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667
Tony Banks          2005            14  14.000000       25  25.000000         173.0  173.000000           1  1.000000           2.0  2.000000
Charlie Batch       2005            23   7.666667       36  12.000000         246.0   82.000000           1  0.333333           1.0  0.333333

Using this information, we want to create 2 more summaries: Completion percentage and TD-Interception ratio. We want to compute the ratio only where the denominator (total interception) is not 0. 


In [68]:
qb_summary["completion_percentage"] = (
    qb_summary[("completions", "sum")] / qb_summary[("attempts", "sum")]
)

qb_summary["td_int_ratio"] = (
    qb_summary[("passing_tds", "sum")] / qb_summary[("interceptions", "sum")]
    .where(qb_summary[("interceptions", "sum")] != 0)
)


/home/jupyter-bchen29@ncsu.edu/.local/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [69]:
qb_summary.head()

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Jeff Blake          2005             8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.000000              0.888889          NaN
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286              0.643519     0.500000
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667              0.533569     1.538462
Tony Banks          2005            14  14.000000       25  25.000000         173.0  173.000000           1  1.000000           2.0  2.000000              0.560000     0.500000
Charlie Batch       2005            23   7.666667       36  12.000000         246.0   82.000000           1  0.333333           1.0  0.333333              0.638889     1.000000

I wonder why Blake have 0 interceptions for 2005! 

Now, let's see who have at least 50 attempts. 


In [70]:
attempt_50 = qb_summary[qb_summary[("attempts", "sum")] >= 50]
attempt_50.head()

/home/jupyter-bchen29@ncsu.edu/.local/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286              0.643519     0.500000
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667              0.533569     1.538462
Jeff Garcia         2005           102  17.000000      173  28.833333         937.0  156.166667           3  0.500000           6.0  1.000000              0.589595     0.500000
Brett Favre         2005           372  23.250000      607  37.937500        3881.0  242.562500          20  1.250000          29.0  1.812500              0.612850     0.689655
Todd Bouman         2005            68  13.600000      122  24.400000         722.0  144.400000           2  0.400000           7.0  1.400000              0.557377     0.285714

Let's sort the rows descending by completion percentage and report the first 40 players. Notice that there are `NaN` values because of a 0 in the denominator. We are not interesting in those, so exclude them!

In [80]:
top40_completion = attempt_50[
    attempt_50["completion_percentage"].notna()
].sort_values(
    by="completion_percentage",
    ascending=False
).head(40)

top40_completion

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
C.J. Beathard       2023            40   6.666667       53   8.833333         349.0   58.166667           1  0.166667           0.0  0.000000              0.754717          NaN
Colt McCoy          2021            74  10.571429       99  14.142857         740.0  105.714286           3  0.428571           1.0  0.142857              0.747475     3.000000
Matt Schaub         2019            50  10.000000       67  13.400000         580.0  116.000000           3  0.600000           1.0  0.200000              0.746269     3.000000
Drew Brees          2018           364  24.266667      489  32.600000        3992.0  266.133333          32  2.133333           5.0  0.333333              0.744376     6.400000
                    2019           281  25.545455      378  34.363636        2979.0  270.818182          27  2.454545           4.0  0.363636              0.743386     6.750000
Mason Rudolph       2023            55  13.750000       74  18.500000         719.0  179.750000           3  0.750000           0.0  0.000000              0.743243          NaN
Taysom Hill         2020            88   5.500000      121   7.562500         928.0   58.000000           4  0.250000           2.0  0.125000              0.727273     2.000000
Nick Foles          2018           141  28.200000      195  39.000000        1413.0  282.600000           7  1.400000           4.0  0.800000              0.723077     1.750000
Drew Brees          2017           386  24.125000      536  33.500000        4334.0  270.875000          23  1.437500           8.0  0.500000              0.720149     2.875000
Sam Bradford        2016           395  26.333333      552  36.800000        3877.0  258.466667          20  1.333333           5.0  0.333333              0.715580     4.000000
Drew Brees          2011           471  29.437500      660  41.250000        5535.0  345.937500          46  2.875000          14.0  0.875000              0.713636     3.285714
Colt McCoy          2014            91  18.200000      128  25.600000        1057.0  211.400000           4  0.800000           3.0  0.600000              0.710938     1.333333
Aaron Rodgers       2020           372  23.250000      526  32.875000        4299.0  268.687500          48  3.000000           5.0  0.312500              0.707224     9.600000
Bailey Zappe        2022            65  16.250000       92  23.000000         781.0  195.250000           5  1.250000           3.0  0.750000              0.706522     1.666667
Drew Brees          2009           363  24.200000      514  34.266667        4388.0  292.533333          34  2.266667          11.0  0.733333              0.706226     3.090909
                    2020           275  22.916667      390  32.500000        2942.0  245.166667          24  2.000000           6.0  0.500000              0.705128     4.000000
Joe Burrow          2021           366  22.875000      520  32.500000        4611.0  288.187500          34  2.125000          14.0  0.875000              0.703846     2.428571
Derek Carr          2019           361  22.562500      513  32.062500        4054.0  253.375000          21  1.312500           8.0  0.500000              0.703704     2.625000
Jake Browning       2023           171  19.000000      243  27.000000        1936.0  215.111111          12  1.333333           7.0  0.777778              0.703704     1.714286
Chase Daniel        2019            45  15.000000       64  21.333333         435.0  145.000000           3  1.000000           2.0  

Do the same for TD-Intercept ratio, and see top 40 players. 

In [81]:
top40_td_int = attempt_50[
    attempt_50["td_int_ratio"].notna()
].sort_values(
    by="td_int_ratio",
    ascending=False
).head(40)

top40_td_int

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Tom Brady           2016           291  24.250000      432  36.000000        3554.0  296.166667          28  2.333333           2.0  0.166667              0.673611    14.000000
Nick Foles          2013           203  15.615385      317  24.384615        2891.0  222.384615          27  2.076923           2.0  0.153846              0.640379    13.500000
Josh McCown         2013           149  18.625000      224  28.000000        1829.0  228.625000          13  1.625000           1.0  0.125000              0.665179    13.000000
Aaron Rodgers       2018           372  23.250000      597  37.312500        4442.0  277.625000          25  1.562500           2.0  0.125000              0.623116    12.500000
Damon Huard         2006           148  14.800000      244  24.400000        1878.0  187.800000          11  1.100000           1.0  0.100000              0.606557    11.000000
Aaron Rodgers       2020           372  23.250000      526  32.875000        4299.0  268.687500          48  3.000000           5.0  0.312500              0.707224     9.600000
                    2021           366  22.875000      531  33.187500        4115.0  257.187500          37  2.312500           4.0  0.250000              0.689266     9.250000
Tom Brady           2010           324  20.250000      492  30.750000        3900.0  243.750000          36  2.250000           4.0  0.250000              0.658537     9.000000
Jake Delhomme       2007            55  18.333333       86  28.666667         617.0  205.666667           8  2.666667           1.0  0.333333              0.639535     8.000000
Aaron Rodgers       2014           341  21.312500      520  32.500000        4381.0  273.812500          38  2.375000           5.0  0.312500              0.655769     7.600000
                    2011           342  22.800000      501  33.400000        4636.0  309.066667          45  3.000000           6.0  0.400000              0.682635     7.500000
Drew Brees          2019           281  25.545455      378  34.363636        2979.0  270.818182          27  2.454545           4.0  0.363636              0.743386     6.750000
Aaron Rodgers       2019           353  22.062500      569  35.562500        4002.0  250.125000          26  1.625000           4.0  0.250000              0.620387     6.500000
Drew Brees          2018           364  24.266667      489  32.600000        3992.0  266.133333          32  2.133333           5.0  0.333333              0.744376     6.400000
Patrick Mahomes     2020           390  26.000000      588  39.200000        4740.0  316.000000          38  2.533333           6.0  0.400000              0.663265     6.333333
Tom Brady           2007           398  24.875000      578  36.125000        4806.0  300.375000          50  3.125000           8.0  0.500000              0.688581     6.250000
Russell Wilson      2019           341  21.312500      516  32.250000        4110.0  256.875000          31  1.937500           5.0  0.312500              0.660853     6.200000
David Garrard       2007           208  17.333333      325  27.083333        2509.0  209.083333          18  1.500000           3.0  0.250000              0.640000     6.000000
Matthew Stafford    2010            57  14.250000       96  24.000000         535.0  133.750000           6  1.500000           1.0  0.250000              0.593750     6.000000
Lamar Jackson       2019           265  17.666667      401  26.733333        3127.0  208.466667          36  2.400000           6.0  